# ContextLens free GPU benchmark

Use Kaggle with Internet enabled and a free GPU accelerator, or Colab with a free GPU runtime. Python 3.12+ is required. This notebook embeds the current local source snapshot, including architecture fixes; no push is required.

Runs three source-read tasks with three repeats, exact o200k_base token counts, parse validation, and exact recovery. Repeats measure latency; this does not measure agent output quality or total provider usage. No paid API keys, Modal account, or paid runtime are used.

This version explicitly uses the experimental memory-efficient T4 attention variant. Numerical equivalence and production output quality are unverified. See docs/benchmark-audit.md in the embedded snapshot.


In [ ]:
import base64, hashlib, io, os, subprocess, sys, zipfile
from pathlib import Path
payload = "..."
data = base64.b64decode(payload)
assert hashlib.sha256(data).hexdigest() == "8a9b06f93d9eba8bb00a22bcab4e76692f487de4fd4d036f6fe05b51e6e3001d"
root = Path.cwd() / "contextlens-benchmark"
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(root)
os.chdir(root)
# A local Git snapshot supplies provenance to the benchmark harness.
subprocess.run(["git", "init"], check=True)
subprocess.run(["git", "add", "."], check=True)
subprocess.run(["git", "-c", "user.name=Benchmark", "-c", "user.email=benchmark@example.invalid", "commit", "-m", "Uploaded ContextLens audit snapshot"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[benchmark]"], check=True)
import torch
assert torch.cuda.is_available(), "Select a free GPU runtime before continuing."
print(torch.cuda.get_device_name())


In [ ]:
import subprocess, sys, json
from pathlib import Path
subprocess.run([sys.executable, "-m", "benchmarks.pruning_quality", "--output", "results/structural.json"], check=True)
try:
    completed = subprocess.run([sys.executable, "-m", "benchmarks.pruning_t4", "--repeats", "3", "--output", "results/runtime.json"], timeout=1200)
    print("Benchmark exit status:", completed.returncode)
except subprocess.TimeoutExpired:
    Path("results/timeout.json").write_text(json.dumps({"status":"timed_out", "seconds":1200, "production_savings_claim":None}))
if Path("results/runtime.json").exists():
    report = json.loads(Path("results/runtime.json").read_text())
    print(json.dumps(report.get("summary", {"status":report["status"]}), indent=2))


In [ ]:
# Optional exploratory quality smoke: three post-hoc questions, not an agent eval.
completed = subprocess.run([sys.executable, "-m", "benchmarks.pruning_qa", "--pruning-report", "results/runtime.json", "--output", "results/qa.json"], capture_output=True, text=True, timeout=360)
print(completed.stdout)
print(completed.stderr[-2000:])
print("Quality smoke exit status:", completed.returncode)


In [ ]:
import shutil
from pathlib import Path
archive = shutil.make_archive(str(Path.cwd().parent / "contextlens-results"), "zip", "results")
print("Results:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))
